# EEG motor imagery: CSP and EEGNet

This experiment uses BCI Competition IV 2a, also called BNCI 2014-001.
For each person, we train on session T and predict left hand, right hand,
feet, or tongue imagery in session E. Each person gets a separate model.

Select the **neuroAI** kernel, then run the cells in order. The first run uses
subject 1; change `SUBJECTS` to `list(range(1, 10))` for the complete experiment.
The code reads the local MATLAB files and does not use MOABB's downloader.

The [project report](REPORT.md) explains the signals, array shapes, and evaluation.
This is an offline, fixed-window experiment; the original competition also required
continuous causal predictions, which this notebook does not implement.

In [ ]:
from pathlib import Path
import copy
import hashlib
import importlib.metadata
import json
import platform
import random
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import welch
import mne
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits
import torch
from torch import nn
from IPython.display import display

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU')
torch.set_num_threads(4)
mne.set_log_level('WARNING')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})

## 1. Experiment settings

`PROJECT_DIR` normally resolves to the folder you opened. If the notebook starts
elsewhere, set it to the absolute project path. Output folders have unique names
so separate experiments keep their own metrics and settings.

The artifact switch drops trials marked by the dataset's expert review from
both sessions. Both models see exactly the same remaining trials. We do not use
the three eye channels as classifier inputs.

In [ ]:
PROJECT_DIR = Path.cwd()
# PROJECT_DIR = Path(r'C:\Users\muzz\Documents\Research Work\EEG motor-imagery classification')
DATA_DIR = PROJECT_DIR / 'BCICIV 2a Motor Imagery EEG Dataset'
SUBJECTS = [1]  # Full experiment: list(range(1, 10))
SEED = 42
LOW_HZ, HIGH_HZ = 8.0, 30.0
CUE_OFFSET = 2.0  # MAT trial markers are trial starts, before the cue.
TMIN, TMAX = 0.5, 3.5  # Seconds after the cue; endpoint is included.
DROP_ARTIFACTS = True
CSP_COMPONENTS = 6
MAX_EPOCHS = 300
PATIENCE = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5
VAL_RUN = 6  # Runs 1-5 select weights; run 6 selects the epoch count.

CLASS_NAMES = ['left hand', 'right hand', 'feet', 'tongue']
CHANNEL_NAMES = ['Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3',
                 'C1', 'Cz', 'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz',
                 'CP2', 'CP4', 'P1', 'Pz', 'P2', 'POz']
assert SUBJECTS and len(set(SUBJECTS)) == len(SUBJECTS)
assert all(s in range(1, 10) for s in SUBJECTS)
assert 0 <= TMIN < TMAX <= 4
assert 0 < LOW_HZ < HIGH_HZ < 125
assert MAX_EPOCHS > 0 and PATIENCE > 0
missing = [DATA_DIR / f'A{s:02d}{session}.mat'
           for s in SUBJECTS for session in ['T', 'E']
           if not (DATA_DIR / f'A{s:02d}{session}.mat').is_file()]
if missing:
    raise FileNotFoundError('Missing local session files: ' + ', '.join(p.name for p in missing)
                            + '. Check DATA_DIR. Official links are in REPORT.md.')

RUN_DIR = PROJECT_DIR / 'results' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
for folder in ['plots', 'histories', 'models']:
    (RUN_DIR / folder).mkdir(parents=True, exist_ok=True)
CONFIG = {name: globals()[name] for name in [
    'SUBJECTS', 'SEED', 'LOW_HZ', 'HIGH_HZ', 'CUE_OFFSET', 'TMIN', 'TMAX',
    'DROP_ARTIFACTS', 'CSP_COMPONENTS', 'MAX_EPOCHS', 'PATIENCE', 'BATCH_SIZE',
    'LEARNING_RATE', 'WEIGHT_DECAY', 'DROPOUT', 'VAL_RUN']}
CONFIG.update({'protocol': 'within_subject_T_to_E', 'class_names': CLASS_NAMES,
               'channels': CHANNEL_NAMES, 'filter': 'Butterworth order 4, zero-phase IIR',
               'refit': 'all T trials for epoch count selected on T run 6',
               'created_utc': datetime.now(timezone.utc).isoformat(),
               'device': str(DEVICE), 'python': platform.python_version(),
               'packages': {p: importlib.metadata.version(p) for p in
                            ['numpy', 'scipy', 'mne', 'scikit-learn', 'torch',
                             'matplotlib', 'pandas']}})
(RUN_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('Results:', RUN_DIR)

## 2. Read and preprocess the local sessions

The MATLAB `data` field contains several continuous runs. The initial eye
calibration blocks have no imagery trials, so we skip them. In each imagery run,
`X` is samples × 25 channels, `trial` holds 1-based trial starts, `y` holds labels
1–4, and `artifacts` holds the review flags.

We convert microvolts to volts for MNE, filter each continuous run independently,
then epoch it. A marker at sample `s` becomes `s - 1 + 2 * 250` for the cue.
At 250 Hz, the inclusive interval 0.5–3.5 s has **751 samples**. Filtering runs
individually also keeps the validation run separate from the fitting runs.

No baseline correction, ICA, rereferencing, or resampling is applied. Filtering
uses future samples within a run, so these results describe offline analysis.

In [ ]:
def load_session(subject, session):
    path = DATA_DIR / f'A{subject:02d}{session}.mat'
    runs = loadmat(path, simplify_cells=True)['data']
    if isinstance(runs, dict):
        runs = [runs]
    all_x, all_y, all_meta, audit = [], [], [], []
    run_number = 0
    for block in runs:
        starts = np.asarray(block['trial']).reshape(-1)
        if starts.size == 0:
            continue
        run_number += 1
        starts = starts.astype(np.int64) - 1
        labels = np.asarray(block['y']).reshape(-1).astype(np.int64)
        flags = np.asarray(block['artifacts']).reshape(-1).astype(np.int64)
        fs = float(block['fs'])
        signal = np.asarray(block['X'], dtype=np.float64)
        if fs != 250 or signal.ndim != 2 or signal.shape[1] != 25:
            raise ValueError(f'Unexpected EEG shape or sampling rate: {path.name}')
        if len(starts) != 48 or len(labels) != 48 or len(flags) != 48:
            raise ValueError(f'Expected 48 labeled trials in {path.name}, run {run_number}')
        if not np.isin(labels, [1, 2, 3, 4]).all():
            raise ValueError(f'Missing/invalid class labels in {path.name}')
        if not np.isin(flags, [0, 1]).all():
            raise ValueError('Artifact flags must be 0 or 1')
        eeg = signal[:, :22].T * 1e-6
        if not np.isfinite(eeg).all():
            raise ValueError(f'Nonfinite EEG samples in {path.name}, run {run_number}')
        cue_samples = starts + int(round(CUE_OFFSET * fs))
        if np.any(starts < 0) or np.any(cue_samples + round(TMAX * fs) >= eeg.shape[1]):
            raise ValueError('Epoch window exceeds the run boundaries')
        info = mne.create_info(CHANNEL_NAMES, sfreq=fs, ch_types='eeg')
        raw = mne.io.RawArray(eeg, info, verbose=False)
        raw.set_montage('standard_1020')
        raw.filter(LOW_HZ, HIGH_HZ, method='iir',
                   iir_params={'order': 4, 'ftype': 'butter'}, phase='zero', verbose=False)
        keep = flags == 0 if DROP_ARTIFACTS else np.ones(len(flags), dtype=bool)
        events = np.column_stack([cue_samples[keep], np.zeros(keep.sum(), dtype=int), labels[keep]])
        epochs = mne.Epochs(raw, events, event_id=None, tmin=TMIN, tmax=TMAX,
                            baseline=None, preload=True, proj=False, reject=None,
                            reject_by_annotation=False, verbose=False)
        x = epochs.get_data(copy=True).astype(np.float32)
        expected_samples = round((TMAX - TMIN) * fs) + 1
        if x.shape != (int(keep.sum()), 22, expected_samples):
            raise ValueError(f'Unexpected epoch shape: {x.shape}')
        all_x.append(x)
        all_y.append(labels[keep] - 1)
        all_meta.append(pd.DataFrame({'subject': subject, 'session': session,
            'run': run_number, 'trial_in_run': np.flatnonzero(keep) + 1,
            'cue_sample': cue_samples[keep], 'artifact': flags[keep]}))
        audit.append({'subject': subject, 'session': session, 'run': run_number,
                      'total_trials': len(flags), 'flagged_trials': int(flags.sum()),
                      'kept_trials': int(keep.sum())})
    if run_number != 6:
        raise ValueError(f'Expected 6 imagery runs in {path.name}; got {run_number}')
    X, y = np.concatenate(all_x), np.concatenate(all_y)
    meta = pd.concat(all_meta, ignore_index=True)
    assert X.shape[0] == len(y) == len(meta)
    assert set(np.unique(y)) == {0, 1, 2, 3}
    return X, y, meta, pd.DataFrame(audit)

preview_subject = SUBJECTS[0]
X_preview, y_preview, meta_preview, audit_preview = load_session(preview_subject, 'T')
print('X:', X_preview.shape, '(trials, EEG channels, samples); unit: volts')
print('y:', y_preview.shape, 'classes:', np.unique(y_preview))
display(audit_preview)

## 3. Look at one trial

C3, Cz, and C4 sit over central scalp regions used in motor-imagery work.
The trace shows the filtered signal in microvolts. The power spectrum averages
all retained training trials at C3, including all four classes. It describes the
filtered input; a peak or difference alone does not establish a reliable decoder.

In [ ]:
trial = 0
times = TMIN + np.arange(X_preview.shape[-1]) / 250
fig, axes = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
for name in ['C3', 'Cz', 'C4']:
    axes[0].plot(times, X_preview[trial, CHANNEL_NAMES.index(name)] * 1e6,
                 label=name, linewidth=0.8)
axes[0].set(xlabel='Seconds after cue', ylabel='EEG (µV)',
             title=f'A{preview_subject:02d}T, trial {trial + 1}: {CLASS_NAMES[y_preview[trial]]}')
axes[0].legend(ncol=3)
freqs, power = welch(X_preview[:, CHANNEL_NAMES.index('C3')] * 1e6, fs=250, nperseg=500, axis=-1)
axes[1].semilogy(freqs, power.mean(axis=0))
axes[1].axvspan(LOW_HZ, HIGH_HZ, alpha=0.12, color='teal')
axes[1].set(xlim=(1, 60), xlabel='Frequency (Hz)', ylabel='Power (µV²/Hz)',
             title='C3 power spectrum, averaged across training trials')
fig.savefig(RUN_DIR / 'plots' / 'signal_preview.png', dpi=150)
plt.show()
plt.close(fig)

## 4. CSP + LDA baseline

CSP learns weighted mixtures of the electrode signals whose variances help
separate the classes. MNE supports multiclass CSP. We retain six log-power
features, then fit shrinkage LDA. The pipeline fits both stages on T only.
These component and classifier settings are fixed before evaluation.

The [MNE CSP example](https://mne.tools/stable/auto_examples/decoding/decoding_csp_eeg.html)
shows the same family of methods on another motor-imagery dataset.

In [ ]:
def make_baseline():
    return Pipeline([
        ('csp', CSP(n_components=CSP_COMPONENTS, reg='ledoit_wolf', log=True,
                    norm_trace=False, cov_est='concat', transform_into='average_power')),
        ('lda', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
    ])

## 5. EEGNet in PyTorch

The temporal convolution mixes nearby samples separately at each electrode.
The grouped spatial convolution then spans all 22 electrodes and learns two
spatial filters for each of eight temporal filters. A depthwise temporal layer
and a 1×1 convolution form the separable block. Pooling reduces the time axis;
dropout discourages reliance on individual activations.

This implementation follows the EEGNet-8,2 structure, with 64- and 16-sample
temporal kernels, ELU, and average pooling by 4 and 8. It adds Adam weight decay.
It returns four logits: PyTorch's cross-entropy loss includes the probability
calculation during training. `argmax` chooses the predicted class.

See the [EEGNet paper](https://arxiv.org/abs/1611.08024) and
[authors' implementation](https://github.com/vlawhern/arl-eegmodels).

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, n_channels=22, n_samples=751, n_classes=4, dropout=0.5):
        super().__init__()
        f1, depth, f2 = 8, 2, 16
        self.temporal = nn.Sequential(
            nn.ZeroPad2d((31, 32, 0, 0)),
            nn.Conv2d(1, f1, (1, 64), bias=False),
            nn.BatchNorm2d(f1, eps=1e-3, momentum=0.01))
        self.spatial = nn.Conv2d(f1, f1 * depth, (n_channels, 1), groups=f1, bias=False)
        self.block1 = nn.Sequential(
            nn.BatchNorm2d(f1 * depth, eps=1e-3, momentum=0.01),
            nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout))
        self.block2 = nn.Sequential(
            nn.ZeroPad2d((7, 8, 0, 0)),
            nn.Conv2d(f1 * depth, f1 * depth, (1, 16), groups=f1 * depth, bias=False),
            nn.Conv2d(f1 * depth, f2, (1, 1), bias=False),
            nn.BatchNorm2d(f2, eps=1e-3, momentum=0.01),
            nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dropout))
        self.classifier = nn.Linear(f2 * (n_samples // 4 // 8), n_classes)

    def forward(self, x):
        x = self.temporal(x)
        x = self.block1(self.spatial(x))
        x = self.block2(x)
        return self.classifier(x.flatten(1))

    @torch.no_grad()
    def constrain_weights(self):
        # Max-norm constraints from the EEGNet implementation.
        for weight, limit in [(self.spatial.weight, 1.0), (self.classifier.weight, 0.25)]:
            norms = weight.norm(2, dim=tuple(range(1, weight.ndim)), keepdim=True)
            weight.mul_((limit / norms.clamp_min(1e-8)).clamp(max=1.0))

example_model = EEGNet(n_samples=X_preview.shape[-1], dropout=DROPOUT)
example_model.eval()
with torch.no_grad():
    example_logits = example_model(torch.zeros(2, 1, 22, X_preview.shape[-1]))
assert example_logits.shape == (2, 4)
print('Trainable parameters:', sum(p.numel() for p in example_model.parameters()))
print('Input: (batch, 1, 22, 751). Output:', tuple(example_logits.shape))

## 6. Train without using E for model selection

We fit on T runs 1–5 and measure validation loss on T run 6. If validation loss
does not improve for 50 epochs, selection stops. The epoch with the lowest
validation loss determines how many epochs to train a fresh model on **all T**.
Only then do we predict E. The curves describe the selection stage, not the
fresh final model.

Each channel uses one mean and standard deviation computed over fitting trials
and time samples. Validation uses the runs 1–5 statistics. Final training and E
use the all-T statistics. Test values never affect normalization.

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def fit_normalizer(X):
    mean = X.mean(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)
    std = X.std(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)
    return mean, np.maximum(std, 1e-12)

def to_tensor(X, mean, std):
    z = ((X - mean) / std).astype(np.float32)
    return torch.as_tensor(z[:, None], device=DEVICE)

def measure(model, X, y):
    model.eval()
    loss_sum, correct = 0.0, 0
    with torch.no_grad():
        for start in range(0, len(y), BATCH_SIZE):
            xb, yb = X[start:start + BATCH_SIZE], y[start:start + BATCH_SIZE]
            logits = model(xb)
            loss_sum += nn.functional.cross_entropy(logits, yb, reduction='sum').item()
            correct += (logits.argmax(1) == yb).sum().item()
    return loss_sum / len(y), correct / len(y)

def train_epoch(model, optimizer, X, y):
    model.train()
    order = torch.randperm(len(y), device=DEVICE)
    loss_sum, correct = 0.0, 0
    for start in range(0, len(y), BATCH_SIZE):
        ids = order[start:start + BATCH_SIZE]
        xb, yb = X[ids], y[ids]
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = nn.functional.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()
        model.constrain_weights()
        loss_sum += loss.item() * len(ids)
        correct += (logits.argmax(1) == yb).sum().item()
    # Training measurements include dropout; validation measurements do not.
    return loss_sum / len(y), correct / len(y)

def fit_eegnet(X, y, meta, subject):
    is_val = meta['run'].to_numpy() == VAL_RUN
    fit_ids, val_ids = np.flatnonzero(~is_val), np.flatnonzero(is_val)
    assert len(fit_ids) and len(val_ids)
    assert set(meta.iloc[fit_ids]['run']).isdisjoint(set(meta.iloc[val_ids]['run']))
    assert set(np.unique(y[fit_ids])) == set(np.unique(y[val_ids])) == {0, 1, 2, 3}
    mean, std = fit_normalizer(X[fit_ids])
    X_fit = to_tensor(X[fit_ids], mean, std)
    X_val = to_tensor(X[val_ids], mean, std)
    y_fit = torch.as_tensor(y[fit_ids], dtype=torch.long, device=DEVICE)
    y_val = torch.as_tensor(y[val_ids], dtype=torch.long, device=DEVICE)
    subject_seed = SEED + subject
    seed_everything(subject_seed)
    model = EEGNet(n_samples=X.shape[-1], dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    best_loss, best_epoch, stale = float('inf'), 0, 0
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = train_epoch(model, optimizer, X_fit, y_fit)
        val_loss, val_acc = measure(model, X_val, y_val)
        if not np.isfinite([train_loss, val_loss]).all():
            raise RuntimeError('Nonfinite loss; inspect the data and learning rate')
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_accuracy': train_acc, 'val_accuracy': val_acc})
        if val_loss < best_loss - 1e-4:
            best_loss, best_epoch, stale = val_loss, epoch, 0
        else:
            stale += 1
        if epoch == 1 or epoch % 25 == 0:
            print(f'A{subject:02d} selection {epoch:3d}: train={train_acc:.3f}, '
                  f'val={val_acc:.3f}, val loss={val_loss:.3f}', flush=True)
        if stale >= PATIENCE:
            break
    del model, optimizer, X_fit, X_val, y_fit, y_val
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    mean, std = fit_normalizer(X)
    X_all = to_tensor(X, mean, std)
    y_all = torch.as_tensor(y, dtype=torch.long, device=DEVICE)
    seed_everything(subject_seed)  # Fresh initialization, fixed epoch budget.
    final_model = EEGNet(n_samples=X.shape[-1], dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    for epoch in range(1, best_epoch + 1):
        train_epoch(final_model, optimizer, X_all, y_all)
        if epoch % 50 == 0:
            print(f'A{subject:02d} all-T refit: {epoch}/{best_epoch}', flush=True)
    selection = {'best_epoch': best_epoch, 'selection_epochs': len(history),
                 'validation_loss': best_loss,
                 'validation_accuracy': history[best_epoch - 1]['val_accuracy'],
                 'selection_fit_trials': len(fit_ids), 'validation_trials': len(val_ids)}
    return final_model, mean, std, pd.DataFrame(history), selection

def predict_eegnet(model, X, mean, std):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(X), BATCH_SIZE):
            logits = model(to_tensor(X[start:start + BATCH_SIZE], mean, std))
            predictions.append(logits.argmax(1).cpu().numpy())
    return np.concatenate(predictions)

## 7. Run the experiment and save results

This cell performs the training. It saves metrics after each subject, along with
trial identities, predictions, filtering counts, model-selection curves, and a
checkpoint for each final EEGNet model. The checkpoint includes normalization
statistics and configuration; it does not include EEG data. Baseline models can
be recreated by fitting the pipeline on T again.

Accuracy measures correct trials. Kappa adjusts agreement for the class
frequencies: `(observed agreement - expected agreement) / (1 - expected agreement)`.
Chance accuracy is 25% for balanced four-class trials; artifact removal can make
the retained class counts uneven. We report the majority-class dummy trained
on T as a separate reference.

In [ ]:
def save_curves(history, subject, best_epoch):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for ax, metric in zip(axes, ['loss', 'accuracy']):
        ax.plot(history.epoch, history[f'train_{metric}'], label='T runs 1–5')
        ax.plot(history.epoch, history[f'val_{metric}'], label='T run 6')
        ax.axvline(best_epoch, color='gray', linestyle=':', label='Selected epoch')
        ax.set(xlabel='Epoch', ylabel=metric.capitalize())
        ax.legend(fontsize=8)
    axes[1].set_ylim(0, 1)
    fig.suptitle(f'A{subject:02d}: EEGNet model selection')
    fig.savefig(RUN_DIR / 'plots' / f'A{subject:02d}_training_curves.png', dpi=150)
    plt.show()
    plt.close(fig)

def save_confusions(y, predictions, subject):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    for ax, (name, pred) in zip(axes, predictions.items()):
        cm = confusion_matrix(y, pred, labels=np.arange(4))
        ax.imshow(cm, cmap='Blues', vmin=0)
        for row in range(4):
            for col in range(4):
                ax.text(col, row, str(cm[row, col]), ha='center', va='center',
                        color='white' if cm[row, col] > cm.max() / 2 else 'black')
        ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES,
               yticklabels=CLASS_NAMES, xlabel='Predicted class', ylabel='True class', title=name)
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
        pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
            RUN_DIR / f'A{subject:02d}_{name.lower().replace("+", "_")}_confusion.csv')
    fig.suptitle(f'A{subject:02d}: evaluation session, trial counts')
    fig.savefig(RUN_DIR / 'plots' / f'A{subject:02d}_confusion.png', dpi=150)
    plt.show()
    plt.close(fig)

metric_rows, audit_frames, selection_rows, prediction_frames = [], [], [], []
file_records = []
experiment_start = time.perf_counter()
for subject in SUBJECTS:
    subject_start = time.perf_counter()
    X_train, y_train, meta_train, audit_train = load_session(subject, 'T')
    # E labels stay in the evaluation section; they never enter fitting functions.
    X_test, y_test, meta_test, audit_test = load_session(subject, 'E')
    audit_frames.extend([audit_train, audit_test])
    trial_identity = pd.concat([meta_train, meta_test], ignore_index=True)
    trial_identity.to_csv(RUN_DIR / f'A{subject:02d}_trial_index.csv', index=False)
    for session in ['T', 'E']:
        path = DATA_DIR / f'A{subject:02d}{session}.mat'
        file_records.append({'file': path.name, 'bytes': path.stat().st_size,
                             'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    print(f'\nA{subject:02d}: {len(y_train)} T trials, {len(y_test)} E trials', flush=True)
    majority = np.bincount(y_train, minlength=4).argmax()
    dummy_accuracy = accuracy_score(y_test, np.full_like(y_test, majority))
    baseline_start = time.perf_counter()
    baseline = make_baseline()
    with threadpool_limits(limits=4):
        baseline.fit(X_train.astype(np.float64), y_train)
        pred_csp = baseline.predict(X_test.astype(np.float64))
    baseline_seconds = time.perf_counter() - baseline_start
    cnn_start = time.perf_counter()
    model, mean, std, history, selection = fit_eegnet(X_train, y_train, meta_train, subject)
    pred_cnn = predict_eegnet(model, X_test, mean, std)
    cnn_seconds = time.perf_counter() - cnn_start
    selection_rows.append({'subject': subject, **selection})
    history.to_csv(RUN_DIR / 'histories' / f'A{subject:02d}_selection.csv', index=False)
    torch.save({'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
                'mean': torch.from_numpy(mean), 'std': torch.from_numpy(std),
                'n_samples': X_train.shape[-1], 'config': CONFIG,
                'subject': subject, 'selection': selection},
               RUN_DIR / 'models' / f'A{subject:02d}_eegnet.pt')
    pred_frame = meta_test.copy()
    pred_frame['true_class'] = y_test
    pred_frame['csp_prediction'] = pred_csp
    pred_frame['eegnet_prediction'] = pred_cnn
    prediction_frames.append(pred_frame)
    for name, pred, seconds in [('CSP+LDA', pred_csp, baseline_seconds),
                                ('EEGNet', pred_cnn, cnn_seconds)]:
        metric_rows.append({'subject': subject, 'model': name,
                            'accuracy': accuracy_score(y_test, pred),
                            'kappa': cohen_kappa_score(y_test, pred, labels=np.arange(4)),
                            'n_train': len(y_train), 'n_test': len(y_test),
                            'majority_dummy_accuracy': dummy_accuracy,
                            'fit_predict_seconds': seconds})
    pd.DataFrame(metric_rows).to_csv(RUN_DIR / 'metrics.csv', index=False)
    pd.concat(audit_frames, ignore_index=True).to_csv(RUN_DIR / 'preprocessing_audit.csv', index=False)
    pd.DataFrame(selection_rows).to_csv(RUN_DIR / 'model_selection.csv', index=False)
    pd.concat(prediction_frames, ignore_index=True).to_csv(RUN_DIR / 'predictions.csv', index=False)
    (RUN_DIR / 'input_files.json').write_text(json.dumps(file_records, indent=2), encoding='utf-8')
    save_curves(history, subject, selection['best_epoch'])
    save_confusions(y_test, {'CSP+LDA': pred_csp, 'EEGNet': pred_cnn}, subject)
    print(f'A{subject:02d} finished in {time.perf_counter() - subject_start:.1f} s; '
          f'CSP={accuracy_score(y_test, pred_csp):.3f}, EEGNet={accuracy_score(y_test, pred_cnn):.3f}', flush=True)
    del model, X_train, X_test
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

metrics = pd.DataFrame(metric_rows)
print(f'Total experiment time: {(time.perf_counter() - experiment_start) / 60:.1f} minutes')
display(metrics)

## 8. Compare subjects and inspect mistakes

The summary averages subject scores, giving every person equal weight. The
pooled confusion matrices add trial counts, so subjects with more retained E
trials contribute more. The shaded accuracy line is a balanced-class chance
reference. It is not a statistical significance test.

Avoid tuning settings after looking at E scores and presenting the same E
session as an untouched test set. For a further comparison, choose the settings
with T-only validation or nested cross-validation and reserve fresh test data.

In [ ]:
summary = metrics.groupby('model').agg(
    mean_accuracy=('accuracy', 'mean'), std_accuracy=('accuracy', 'std'),
    mean_kappa=('kappa', 'mean'), std_kappa=('kappa', 'std'),
    subjects=('subject', 'nunique'))
summary.to_csv(RUN_DIR / 'summary.csv')
display(summary)
table = metrics.pivot(index='subject', columns='model', values=['accuracy', 'kappa'])
table.to_csv(RUN_DIR / 'per_subject_comparison.csv')
display(table)

scores = metrics.pivot(index='subject', columns='model', values='accuracy')
ax = scores.plot.bar(figsize=(10, 4), color=['#397b8a', '#d18c42'])
ax.axhline(0.25, color='gray', linestyle='--', label='25% balanced chance')
ax.set(xlabel='Subject', ylabel='Evaluation accuracy', ylim=(0, 1),
       title='Within-subject prediction: session T → session E')
plt.xticks(rotation=0)
ax.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / 'plots' / 'subject_comparison.png', dpi=150)
plt.show()
plt.close(ax.figure)

pooled = pd.concat(prediction_frames, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, (name, column) in zip(axes, [('CSP+LDA', 'csp_prediction'), ('EEGNet', 'eegnet_prediction')]):
    cm = confusion_matrix(pooled.true_class, pooled[column], labels=range(4))
    proportions = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(proportions, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{proportions[i, j]:.0%}\n({cm[i, j]})', ha='center', va='center',
                    color='white' if proportions[i, j] > 0.5 else 'black', fontsize=9)
    ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES,
           yticklabels=CLASS_NAMES, xlabel='Predicted', ylabel='True', title=name)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.suptitle('Pooled E trials: row percentages and counts')
fig.savefig(RUN_DIR / 'plots' / 'pooled_confusion.png', dpi=150)
plt.show()
plt.close(fig)
print('Saved results in:', RUN_DIR)

## 9. Reload a final EEGNet model

This cell loads subject 1 of the selected list from this experiment. Checkpoints
store tensors for the model and normalization. Only load checkpoint files you
trust. The local MATLAB data still needs the same preprocessing as training.

In [ ]:
checkpoint = torch.load(RUN_DIR / 'models' / f'A{SUBJECTS[0]:02d}_eegnet.pt',
                        map_location='cpu', weights_only=True)
loaded_model = EEGNet(n_samples=checkpoint['n_samples'],
                       dropout=checkpoint['config']['DROPOUT']).to(DEVICE)
loaded_model.load_state_dict(checkpoint['state_dict'])
loaded_model.eval()
print('Loaded EEGNet for subject', checkpoint['subject'])
# For preprocessed epochs X_new in volts:
# predicted = predict_eegnet(loaded_model, X_new,
#                            checkpoint['mean'].numpy(), checkpoint['std'].numpy())

## Further reading

Start with the [report](REPORT.md), then try the
[MNE overview](https://mne.tools/stable/auto_tutorials/intro/10_overview.html)
and [CSP tutorial](https://mne.tools/stable/auto_examples/decoding/decoding_csp_eeg.html).
The [dataset description](https://www.bbci.de/competition/iv/desc_2a.pdf) explains
the recording and cues. The [EEGNet paper](https://arxiv.org/abs/1611.08024)
explains the compact CNN and its evaluation across BCI tasks.